# 2.2 Runtime Context — Per-Request User Data

## What you will learn in this notebook

- What **runtime context** is and why it's different from state and memory
- How to define a **context schema** using a Python dataclass
- How to pass context values at **call time** (per-request, not globally)
- How tools access context via **`ToolRuntime`** without it being a tool argument
- Why context is the right pattern for **per-user, per-request data**

---

### Three ways to give an agent data — when to use which

| Mechanism | What it is | Persists? | User sees it? | Best for |
|---|---|---|---|---|
| **Messages** | Conversation history | Yes (via checkpointer) | Yes | Back-and-forth dialogue |
| **State** | Custom fields on the graph state | Yes (via checkpointer) | No | Derived data the agent computes |
| **Runtime Context** | Data injected per `.invoke()` call | No | No | Per-request user profile data |

### What is runtime context?

**Runtime context** is data you pass to the agent at invocation time that is available to tools but is NOT part of the conversation history and does NOT get stored in state.

Think of it like a **request header** in a web API:
- A web request carries `Authorization: Bearer <token>` in the header
- The handler reads the token to identify the user
- The token is not part of the request body; the user never typed it

Similarly, runtime context is like a **session object** your server injects at request time:
```python
# In a FastAPI endpoint:
agent.invoke(user_message, context=UserContext(user_id=current_user.id, plan="pro"))
```

The tools read `runtime.context.user_id` — the user never said their ID, and it's never stored in the conversation.

### Why not just put it in the system prompt?

You could put `favourite_colour: blue` in the system prompt — but:
- System prompts are part of the conversation, consuming tokens every call
- System prompts are static strings; context objects are typed and validated
- Sensitive data (user IDs, payment info) shouldn't be in prompts

In [1]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
config = {
    "a":1,
    "b":2,
}
def add(a, b, config):
    return a + b

print(add(1,2,config))

3


In [3]:
# ============================================================
# CELL 2: Define the Context Schema
# ============================================================
# The context schema is a Python dataclass that defines WHAT data
# can be passed to the agent at invocation time.
#
# We use @dataclass for simplicity — LangChain also accepts Pydantic
# models if you need field validation.
#
# Default values are important:
#   favourite_colour: str = "blue"  → if not supplied, defaults to blue
#   least_favourite_colour: str = "yellow"
#
# In a real app this might be:
#   @dataclass
#   class UserContext:
#       user_id: str
#       subscription_plan: str = "free"
#       preferred_language: str = "en"
#       timezone: str = "UTC"

from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"      # Default if not overridden
    least_favourite_colour: str = "yellow"

In [4]:
# ============================================================
# CELL 3: Create an Agent That Accepts Context (No Tools Yet)
# ============================================================
# context_schema=ColourContext tells LangChain:
#   "When this agent is invoked, expect an optional context=
#    argument that is an instance of ColourContext"
#
# Without context_schema, passing context= to .invoke() is ignored.
# With it, the context object becomes available to tools via ToolRuntime.
#
# At this stage (no tools), the context is declared but the agent
# can't read it — it doesn't have a tool to access it.
# This is just the declaration step.

from langchain.agents import create_agent

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    context_schema=ColourContext   # Declare what context type to expect
)

In [5]:
# ============================================================
# CELL 4: Invoke With Context — But No Tool to Read It
# ============================================================
# We pass context=ColourContext() — the default values (blue/yellow).
#
# The model CANNOT read this context directly — it doesn't appear
# in the conversation history or system prompt. The agent has no
# tool to access it, so it will say it doesn't know the colour.
#
# This demonstrates the important point: context is data for TOOLS,
# not for the model. The model needs a tool to surface the context.
# See Section 2 for that.

from langchain.messages import HumanMessage


response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()   # Inject context at call time
)

In [6]:
# ============================================================
# CELL 5: Inspect — The Agent Doesn't Know (No Tool)
# ============================================================
# The response will say something like "I don't have information
# about your preferences." — because the context is injected but
# inaccessible without a tool.

from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='347510ba-9c0e-4d5a-bfca-9e99af645fa0'),
              AIMessage(content='I’m not sure what your favorite colour is—could you let me know? If you’d like, we can chat about colours you like and why they stand out to you!', additional_kwargs={'reasoning_content': 'We have a user asking "What is my favourite colour?" There\'s no prior context. The user asks a personal question that the assistant doesn\'t know. The assistant should respond that it doesn\'t know, ask for clarification, or something. There\'s no disallowed content. So we can respond politely: "I don\'t know, could you tell me?" Also maybe ask if they\'d like to discuss favourite colours.'}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 77, 'total_tokens': 201, 'completion_time': 0.256048299, 'completion_tokens_details': {'reasoning_tokens': 79}, 'prompt_time': 0.003391615, 'prompt_

---
## Section 2 — Accessing Context From Tools

Tools access context via `ToolRuntime` — a special parameter that LangChain injects automatically when the tool is called.

### The `ToolRuntime` pattern

```python
@tool
def my_tool(user_arg: str, runtime: ToolRuntime) -> str:
    # runtime is injected by LangChain — the model never passes it
    # runtime.context is the ColourContext object we passed to .invoke()
    data = runtime.context.some_field
    return data
```

**Critical**: `runtime: ToolRuntime` must be the **last parameter** and is NOT included in the tool's schema. The model never knows about it — it can't pass it even if it wanted to. LangChain injects it silently.

This is how you pass server-side secrets (user IDs, API tokens) to tools without the model ever seeing them.

In [7]:
# ============================================================
# CELL 6: Define Tools That Read From Context
# ============================================================
# ToolRuntime is the gateway to context inside a tool.
#
# Key rules for ToolRuntime:
#   1. It must be typed as `runtime: ToolRuntime` exactly
#   2. It must be the LAST parameter in the function signature
#   3. It does NOT appear in the tool's argument schema
#      → The model cannot and does not pass this argument
#      → LangChain injects it automatically at tool call time
#
# runtime.context  → the ColourContext object from .invoke(context=...)
# runtime.context.favourite_colour  → "blue" (or whatever was passed)
#
# Notice: these tools take NO user-facing arguments (no `x: float`, etc.)
# They purely read from context. The model calls them with empty args
# and gets back the context value.

from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    # runtime is injected by LangChain — not passed by the model
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [8]:
# ============================================================
# CELL 7: Create the Agent With Context-Reading Tools
# ============================================================

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext   # Must declare the schema to use context
)

In [9]:
# ============================================================
# CELL 8: Invoke With Default Context (blue)
# ============================================================
# ColourContext() uses default values: favourite_colour="blue"
#
# The agent will:
#   1. See the question about favourite colour
#   2. Decide to call get_favourite_colour tool
#   3. LangChain injects runtime.context = ColourContext()
#   4. Tool returns "blue"
#   5. Agent replies: "Your favourite colour is blue"

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()   # Default: favourite_colour="blue"
)

pprint(response)

C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='36fe3646-37e5-4ccc-9a38-fbdcc55453c9'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get favourite colour using function. Use get_favourite_colour.', 'tool_calls': [{'id': 'fc_c86971f9-f979-4b07-9fb4-394ce692c352', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 149, 'total_tokens': 186, 'completion_time': 0.07689837, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.007172286, 'prompt_tokens_details': None, 'queue_time': 0.28831063, 'total_time': 0.084070656}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f73454f048', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f5114-dc40-7e12-bf95-f199fea65520-0', too

In [10]:
# ============================================================
# CELL 9: Invoke With Custom Context (green) — Same Agent!
# ============================================================
# This is the power of runtime context: the SAME agent and the SAME
# tools, but different context per call = different output per user.
#
# ColourContext(favourite_colour="green") overrides the default.
# The tool reads runtime.context.favourite_colour → "green"
# and the agent replies accordingly.
#
# In production: each HTTP request would pass a different UserContext
# (different user_id, subscription plan, locale, etc.) and the same
# agent would serve all users correctly without any shared state
# or per-user agent instances.

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")  # Override for this request
)

pprint(response)

C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='a514346f-403e-4fa6-ad8a-1c68b286eb27'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer user question: "What is my favourite colour?" We have functions to get favourite colour and least favourite colour. So we should call get_favourite_colour.', 'tool_calls': [{'id': 'fc_fe457930-7217-4940-9ccd-0f7bda3ba0bf', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 149, 'total_tokens': 206, 'completion_time': 0.117659385, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.008327731, 'prompt_tokens_details': None, 'queue_time': 0.293231295, 'total_time': 0.125987116}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_93703442d9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls',

---
## Summary

| Concept | Key takeaway |
|---|---|
| **Runtime context** | Per-request data injected at `.invoke()` time — not in history, not in state |
| **`context_schema=`** | Declares the expected context type on `create_agent()` |
| **`context=`** | Passes an instance at `.invoke()` time |
| **`ToolRuntime`** | Injected automatically into tools as the last parameter — model never sees it |
| **`runtime.context`** | Access the context object inside a tool |
| **Isolation** | Same agent, different context per call = correct per-user behaviour |

### When to use runtime context vs alternatives

```
User-typed data ("my name is Seán")  →  Messages / Memory
Agent-computed data ("user is premium")  →  State
Server-side user profile (user_id, plan)  →  Runtime Context  ← this notebook
```

### Next steps
- **2.2 State** — writing and reading custom fields on the agent's graph state
- **2.3 Multi-Agent** — orchestrating multiple specialised agents from a main agent

In [11]:
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage
from pprint import pprint

@tool
def get_user_demo_graphic(user_id:str) -> str:
    """Get a user's demo graphic"""
    return "User is from Bengaluru"

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[get_user_demo_graphic]
)

response = agent.invoke({"messages": [HumanMessage(content="whats the demograhic information of the user with user_id:1234?")]})
pprint(response)

{'messages': [HumanMessage(content='whats the demograhic information of the user with user_id:1234?', additional_kwargs={}, response_metadata={}, id='8d70d58d-5802-4a6b-98b3-71604fda689c'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks for demographic information of user with user_id:1234. We have a tool get_user_demo_graphic. We need to call it.', 'tool_calls': [{'id': 'fc_bda8f9c7-ee5f-422b-877c-2fd1b28b9a2e', 'function': {'arguments': '{"user_id":"1234"}', 'name': 'get_user_demo_graphic'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 141, 'total_tokens': 205, 'completion_time': 0.136926773, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.005680642, 'prompt_tokens_details': None, 'queue_time': 0.283306415, 'total_time': 0.142607415}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_df9620fe21', 'service_tier': 'on_demand', 'finish_reason': 'tool_ca

In [12]:
from dataclasses import dataclass
user_id = "1234"

@dataclass
class UserInfo:
    user_id: str=user_id

@tool
def get_user_demo_graphic(runtime:ToolRuntime) -> str:
    """Get a user's demo graphic"""
    user_id = runtime.context.user_id
    pprint(runtime.state)
    """logic"""
    return f"User:{user_id} is from Bengaluru"

from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(model="groq:openai/gpt-oss-120b",tools=[get_user_demo_graphic],context_schema=UserInfo,checkpointer=InMemorySaver())

# config tells LangGraph which conversation thread this call belongs to
config = {"configurable": {"thread_id": "1"}}


response = agent.invoke({"messages": [HumanMessage(content="whats my demographic information?")]},context=UserInfo(),config=config)
pprint(response)
from langgraph.checkpoint.memory import InMemorySaver
pprint(response["messages"][-1].content)




C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserInfo(user_id='1234'), input_type=UserInfo])
  function=lambda v, h: h(v), schema=original_schema
C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserInfo(user_id='1234'), input_type=UserInfo])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='whats my demographic information?', additional_kwargs={}, response_metadata={}, id='19880c75-8a65-4a1d-9e70-6ba33bcb2d24'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "whats my demographic information?" We need to retrieve user demographic info. There\'s a function get_user_demo_graphic (likely typo "graphic" but it\'s the function). We should call it with no arguments.', 'tool_calls': [{'id': 'fc_239d55d0-09c5-4b57-a0eb-77818fc4ad62', 'function': {'arguments': '{}', 'name': 'get_user_demo_graphic'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 124, 'total_tokens': 193, 'completion_time': 0.142971026, 'completion_tokens_details': {'reasoning_tokens': 47}, 'prompt_time': 0.004923764, 'prompt_tokens_details': None, 'queue_time': 0.280458002, 'total_time': 0.14789479}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'servic

In [13]:
response = agent.invoke({"messages": [HumanMessage(content="where do i live?")]},context=UserInfo(),config=config)
pprint(response["messages"][-1].content)

('You’re based in\u202fBengaluru. If you’d like to confirm or update any '
 'details, just let me know!')


In [14]:
pprint(response)

{'messages': [HumanMessage(content='whats my demographic information?', additional_kwargs={}, response_metadata={}, id='19880c75-8a65-4a1d-9e70-6ba33bcb2d24'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "whats my demographic information?" We need to retrieve user demographic info. There\'s a function get_user_demo_graphic (likely typo "graphic" but it\'s the function). We should call it with no arguments.', 'tool_calls': [{'id': 'fc_239d55d0-09c5-4b57-a0eb-77818fc4ad62', 'function': {'arguments': '{}', 'name': 'get_user_demo_graphic'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 124, 'total_tokens': 193, 'completion_time': 0.142971026, 'completion_tokens_details': {'reasoning_tokens': 47}, 'prompt_time': 0.004923764, 'prompt_tokens_details': None, 'queue_time': 0.280458002, 'total_time': 0.14789479}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'servic

In [15]:
"""
persona_agent.py

A LangChain agent that listens to a conversation, extracts durable facts about
the user (role, company, projects, skills, preferences, interests, location,
goals), and stores them as atomic facts in a semantic (vector) memory store,
keyed by user_id. Facts can later be retrieved by meaning, not just exact
keyword match -- e.g. searching "cloud infra background" will surface a
stored fact like "Works extensively with AWS Bedrock and EKS" even though
the words don't overlap.

user_id is NOT baked into the agent at construction time. The agent is built
once, and user_id is passed in per-call via LangGraph's runtime context
(context_schema=UserInfo), the same way you'd pass it in a multi-tenant
service where one agent instance serves many users. Each tool reads
runtime.context.user_id to know which user's memory to read/write, and a
small in-process cache keeps one PersonaMemoryStore (one Chroma collection)
per user_id so repeated calls for the same user don't reopen the store.

Conversation turns are threaded via LangGraph's checkpointer + thread_id in
config, so multi-turn chat history works independently of persona memory
(persona memory is long-term/cross-session; the checkpointer is short-term/
per-thread).

Two ways to use it:

1. Interactive agent mode (build_persona_agent):
   The agent has three tools -- save_persona_fact, search_persona, and
   get_full_persona -- and decides for itself when to call them. user_id
   comes from runtime context on every invoke() call.

2. Batch extraction mode (extract_persona_from_text / capture_persona):
   Feed it a full transcript (or a single message) and it extracts facts in
   one structured-output call, then stores them directly against a
   user_id -- no agent loop involved. Good for backfilling persona memory
   from existing conversation logs.

Requirements: see requirements.txt
Env vars:
    ANTHROPIC_API_KEY   - required
    PERSONA_AGENT_MODEL - optional, defaults to "claude-sonnet-4-6"
"""

import os
import time
from dataclasses import dataclass
from typing import Dict, List, Optional

from pydantic import BaseModel, Field

#from langchain_anthropic import ChatAnthropic
from langchain_groq import ChatGroq
# from langchain_chroma import Chroma
# from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


DEFAULT_MODEL = os.environ.get("PERSONA_AGENT_MODEL", "qwen/qwen3-32b")
DEFAULT_PERSIST_DIR = os.environ.get("PERSONA_MEMORY_DIR", "./persona_memory_db")


# --------------------------------------------------------------------------
# Runtime context -- this is what carries user_id into every tool call
# --------------------------------------------------------------------------

@dataclass
class UserInfo:
    """
    Passed to agent.invoke(..., context=UserInfo(user_id=...)) on every call.
    Tools read it via runtime.context.user_id. This is how a single agent
    instance serves many users without user_id being baked in at build time.
    """
    user_id: str = "default_user"


# --------------------------------------------------------------------------
# Semantic memory store
# --------------------------------------------------------------------------

# class PersonaMemoryStore:
#     """
#     Wraps a persistent Chroma collection, one per user_id, so multiple users'
#     persona memories never mix. Each stored fact is a short, atomic,
#     third-person sentence embedded independently -- this keeps retrieval
#     precise (searching "diet preferences" won't drag in an unrelated fact
#     about someone's job title just because they're in the same document).
#     """
#
#     def __init__(
#         self,
#         user_id: str,
#         persist_dir: str = DEFAULT_PERSIST_DIR,
#         embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2",
#     ):
#         self.user_id = user_id
#         embeddings = HuggingFaceEmbeddings(model_name=embedding_model)
#         self.vectorstore = Chroma(
#             collection_name=f"persona_{user_id}",
#             embedding_function=embeddings,
#             persist_directory=persist_dir,
#         )
#
#     def save_facts(self, facts: List[str]) -> int:
#         """Embed and store each fact as its own document. Returns count saved."""
#         docs = [
#             Document(
#                 page_content=fact.strip(),
#                 metadata={"user_id": self.user_id, "ts": time.time()},
#             )
#             for fact in facts
#             if fact and fact.strip()
#         ]
#         if docs:
#             self.vectorstore.add_documents(docs)
#         return len(docs)
#
#     def search(self, query: str, k: int = 5) -> List[str]:
#         """Semantic search over stored facts. Returns the k most relevant."""
#         results = self.vectorstore.similarity_search(query, k=k)
#         return [d.page_content for d in results]
#
#     def get_all(self) -> List[str]:
#         """Return every fact currently stored for this user."""
#         raw = self.vectorstore.get()
#         return raw.get("documents", []) or []
#
#     def clear(self) -> None:
#         """Wipe all stored facts for this user. Use with care."""
#         raw = self.vectorstore.get()
#         ids = raw.get("ids", [])
#         if ids:
#             self.vectorstore.delete(ids=ids)


# --------------------------------------------------------------------------
# Per-user store cache
# --------------------------------------------------------------------------
# One agent instance serves many users, so tools can't have a single store
# bound via closure anymore. Instead we lazily open one PersonaMemoryStore
# per user_id and cache it here, keyed by (user_id, persist_dir), so the
# underlying Chroma collection isn't reopened on every tool call.

# _STORE_CACHE: Dict[str, PersonaMemoryStore] = {}
#
#
# def get_store(user_id: str, persist_dir: str = DEFAULT_PERSIST_DIR) -> PersonaMemoryStore:
#     key = f"{persist_dir}::{user_id}"
#     if key not in _STORE_CACHE:
#         _STORE_CACHE[key] = PersonaMemoryStore(user_id=user_id, persist_dir=persist_dir)
#     return _STORE_CACHE[key]


# --------------------------------------------------------------------------
# Tools -- user_id is read from runtime.context on every call, not closed over
# --------------------------------------------------------------------------

@tool
def save_persona_fact(facts: List[str], runtime: ToolRuntime) -> str:
    """
    Save one or more short, atomic, third-person facts about the current
    user to long-term semantic memory. Only call this for durable facts
    (role, company, project, skill, preference, interest, location, goal)
    -- never for small talk, and never for sensitive data (passwords,
    account numbers, health details) unless the user explicitly asks you
    to remember it.

    Example facts: "Works as a Lead Data Architect at Gilead Sciences",
    "Prefers concrete examples over abstract explanations".
    """
    user_id = runtime.context.user_id
    print(user_id,facts)
    # store = get_store(user_id)
    # n = store.save_facts(facts)
    return f"Saved fact(s) to persona memory for user_id={user_id}."


@tool
def search_persona(query: str, runtime: ToolRuntime, k: int = 5) -> str:
    """
    Semantically search the current user's stored persona facts for ones
    relevant to the query. Use this when you need context about the user
    that may not have been mentioned earlier in this conversation.
    """
    user_id = runtime.context.user_id
    # store = get_store(user_id)
    # results = store.search(query, k=k)
    # if not results:
    #     return f"No relevant persona facts found for user_id={user_id}."
    # return "\n".join(f"- {r}" for r in results)
    return ""


# @tool
# def get_full_persona(runtime: ToolRuntime) -> str:
#     """
#     Return every fact currently stored about the current user. Use this
#     when the user asks something like 'what do you know about me?'.
#     """
#     user_id = runtime.context.user_id
#     store = get_store(user_id)
#     facts = store.get_all()
#     if not facts:
#         return f"No persona facts stored yet for user_id={user_id}."
#     return "\n".join(f"- {f}" for f in facts)


PERSONA_TOOLS = [save_persona_fact, search_persona]


# --------------------------------------------------------------------------
# Agent construction (interactive mode)
# --------------------------------------------------------------------------

PERSONA_SYSTEM_PROMPT = """You are a helpful assistant that also maintains a \
long-term persona memory about the user you're talking to.

Rules for using your memory tools:
- Whenever the user reveals a durable fact about themselves -- their role, \
company, project, skills, preferences, interests, location, or goals -- call \
save_persona_fact with the fact(s) rewritten as short, atomic, third-person \
sentences (e.g. "Is based in Rajahmundry, Andhra Pradesh").
- Do not save one-off, transient, or purely conversational content (e.g. \
"asked what time it is").
- Never save sensitive data (passwords, financial account numbers, health \
details) unless the user explicitly asks you to remember it.
- Before assuming you don't know something about the user, call \
search_persona or get_full_persona to check memory first.
- When asked what you know about the user, answer only from what \
get_full_persona / search_persona actually return -- do not invent facts.
- Otherwise, just have a normal, helpful conversation.
"""


def build_persona_agent(model: Optional[str] = None, checkpointer=None):
    """
    Build ONE agent instance meant to be reused across users and sessions.
    user_id is supplied per-call via context, not here -- so this function
    takes no user_id argument.

    Usage:
        agent = build_persona_agent()
        config = {"configurable": {"thread_id": "session-1"}}
        response = agent.invoke(
            {"messages": [HumanMessage(content="I work at Gilead")]},
            context=UserInfo(user_id="1234"),
            config=config,
        )
    """
    llm = ChatGroq(model=model or DEFAULT_MODEL, temperature=0)
    agent = create_agent(
        model=llm,
        tools=PERSONA_TOOLS,
        system_prompt=PERSONA_SYSTEM_PROMPT,
        context_schema=UserInfo,
        checkpointer=checkpointer or InMemorySaver(),
    )
    return agent


# --------------------------------------------------------------------------
# Batch extraction (non-agentic mode, for backfilling from transcripts)
# --------------------------------------------------------------------------

class PersonaExtraction(BaseModel):
    facts: List[str] = Field(
        description=(
            "Atomic, third-person facts about the user learned from the "
            "conversation. Empty list if nothing durable was learned."
        )
    )


def extract_persona_from_text(conversation_text: str, model: Optional[str] = None) -> List[str]:
    """One-shot structured extraction of persona facts from raw text."""
    llm = ChatGroq(model=model or DEFAULT_MODEL, temperature=0)
    structured_llm = llm.with_structured_output(PersonaExtraction)
    result = structured_llm.invoke(
        "Extract durable persona facts (role, company, projects, skills, "
        "interests, preferences, location, goals) about the user from the "
        "conversation below. Ignore small talk and one-off requests. "
        "Write each fact as a short, third-person sentence.\n\n"
        f"Conversation:\n{conversation_text}"
    )
    return result.facts


def capture_persona(
    conversation_text: str,
    user_id: str,
    model: Optional[str] = None,
    persist_dir: str = DEFAULT_PERSIST_DIR,
) -> List[str]:
    """Extract facts from conversation_text and store them against user_id."""
    facts = extract_persona_from_text(conversation_text, model=model)
    # store = get_store(user_id, persist_dir=persist_dir)
    # store.save_facts(facts)
    return facts


# --------------------------------------------------------------------------
# Demo CLI -- one agent, keyed by user_id via context, to show the pattern
# --------------------------------------------------------------------------

if __name__ == "__main__":
    agent = build_persona_agent()

    # One agent instance; user_id flows in per-call via context, not via
    # closures or globals. Different user_id -> different memory, same agent.
    config_a = {"configurable": {"thread_id": "thread-1235"}}
    response = agent.invoke(
        {"messages": [HumanMessage(content="I'm a freelance AI engineer based in Bengaluru.")]},
        context=UserInfo(user_id="1234"),
        config=config_a,
    )
    print("Agent:", response["messages"][-1].content)

    response = agent.invoke(
        {"messages": [HumanMessage(content="What do you know about me?")]},
        context=UserInfo(user_id="1234"),
        config=config_a,
    )
    pprint(response)
    print("Agent:", response["messages"][-1].content)

    # print("\nStored persona facts for user_id=1234:")
    # for fact in get_store("1234").get_all():
    #     print(" -", fact)

C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserInfo(user_id='1234'), input_type=UserInfo])
  function=lambda v, h: h(v), schema=original_schema
C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserInfo(user_id='1234'), input_type=UserInfo])
  return self.__pydantic_serializer__.to_python(


1234 ['Works as a freelance AI engineer.', 'Is based in Bengaluru.']
Agent: I've noted that you're a freelance AI engineer based in Bengaluru. How can I assist you with your current projects or challenges?


BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: attempted to call tool 'get_full_persona' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<tool_call>\n{"name": "get_full_persona", "arguments": {}}\n</tool_call>'}}